# Testing the Simplification of Regular Expressions

The notebook `Simplify.ipynb` provides the function `simplify`, which simplifies a regular expression, and the
function `to_str`, which turns a regular expression into a string that contains only the parentheses that are
needed.

In [ ]:
#use "../lib/nb.ml";;
run_notebook "Simplify.ipynb"

The notebook `RegExp-Parser.ipynb` provides the function `parse`, which turns a string into a regular
expression, and the notebook `09-Equivalence.ipynb` provides the function `regExpEquiv`, which checks whether
two regular expressions are equivalent.  We use it to check that `simplify` does not change the language of a
regular expression.

In [ ]:
run_notebook "RegExp-Parser.ipynb";;
run_notebook "09-Equivalence.ipynb"

**Note:** The function `run_notebook` executes all cells of a notebook, just like the magic command `%run`.  The
Python version has to mark the calls of functions that are defined in the loaded notebooks with
`# type: ignore`.  The OCaml compiler knows these functions, so every call is type checked.

In [ ]:
#show_type regexp

## Examples

Each of the following regular expressions is simplified by one of the laws discussed in the notebook
`Simplify.ipynb`.  The last example describes the complement of the language
$L\bigl((\texttt{a} + \texttt{b})^* \cdot \texttt{a} \cdot \texttt{b} \cdot (\texttt{a} + \texttt{b})^*\bigr)$.  It has been computed in the
notebook `12-Complement.ipynb` and then simplified with rewrite rules that only compare neighbouring summands.

In [ ]:
let sigma = FSet.of_list [ 'a'; 'b' ]
;;
List.iter
  (fun s ->
    let r = parse s in
    let t = simplify r in
    Printf.printf "%s ↦ %s %b\n%!" (ljust (to_str r) 24) (ljust (to_str t) 12) (regExpEquiv r t sigma))
  [ "b + a + b*"; "(𝜀 + a + b)*"; "(a*b*)*"; "ab + aa"; "a + ab*"; "a*(a + 𝜀)"; "b*aa* + 𝜀 + b + b*" ]

## Random Tests

In order to test the function `simplify` more thoroughly, we generate random regular expressions and check that
`simplify` does not change their language.

**Function `random_regexp n rng`**
- *Input:* `n` is a natural number and `rng` is the state of a random number generator.
- *Output:* A random regular expression over the alphabet $\{\texttt{a}, \texttt{b}\}$ that contains exactly `n`
  operators.  Its atoms are chosen from $\texttt{a}$, $\texttt{b}$, $\varepsilon$, and $\emptyset$.

`Random.State.int rng n` returns a random number from the set $\{0, \cdots, n-1\}$.  The left argument of a binary
operator is generated before the right one, as in the Python version.

In [ ]:
let rec random_regexp (n : int) (rng : Random.State.t) : regexp =
  if n = 0 then
    match Random.State.int rng 4 with
    | 0 -> Char 'a'
    | 1 -> Char 'b'
    | 2 -> Epsilon
    | _ -> Empty
  else
    match Random.State.int rng 3 with
    | 0 -> Star (random_regexp (n - 1) rng)
    | 1 ->
        let k     = Random.State.int rng n in
        let left  = random_regexp k rng in
        let right = random_regexp (n - 1 - k) rng in
        Sum (left, right)
    | _ ->
        let k     = Random.State.int rng n in
        let left  = random_regexp k rng in
        let right = random_regexp (n - 1 - k) rng in
        Product (left, right)

We test `simplify` with 3000 random regular expressions that have up to 14 operators.  The random number
generator is initialized with a fixed seed, so that the test is reproducible.  The test prints every regular
expression whose simplification is not equivalent to it, the number of these failures, and the total sizes of
the regular expressions before and after simplification.  Since the random number generator of OCaml differs
from the one of Python, the random regular expressions and hence the total sizes differ from those of the Python
version.  The number of failures has to be $0$ in both versions.

In [ ]:
let rng      = Random.State.make [| 42 |]
let failures = ref 0
let before   = ref 0
let after    = ref 0
;;
for _ = 1 to 3000 do
  let r = random_regexp (1 + Random.State.int rng 14) rng in
  let t = simplify r in
  before := !before + size r;
  after  := !after + size t;
  if not (regExpEquiv r t sigma) then begin
    incr failures;
    Printf.printf "%s ↦ %s\n%!" (to_str r) (to_str t)
  end
done;
Printf.printf "failures: %d, total size before: %d, after: %d\n%!" !failures !before !after